# 🏨 Hotel LuxStay - Concierge Virtual

Chatbot inteligente para atendimento hoteleiro com:
- 🎙️ **Transcrição de áudio** (Whisper)
- 📚 **RAG** - Retrieval-Augmented Generation
- 🌐 **Tradução automática** (8 idiomas)
- 🔊 **Resposta em áudio** (gTTS)

**Problema:** Atendimento repetitivo com baixa eficiência no Hotel LuxStay.
**Solução:** Chatbot com RAG, voz e tradução para automatizar respostas frequentes.

---


## 1. Instalação das Dependências


In [ ]:
!pip install -q groq gTTS openai-whisper deep-translator scikit-learn numpy


## 2. Configuração da API Key

Pegue sua chave **grátis** em: https://console.groq.com

**Opção recomendada:** Use os Secrets do Colab (ícone de 🔑 na barra lateral esquerda). Adicione uma secret com nome `GROQ_API_KEY`.


In [ ]:
import os
from google.colab import userdata

# Opção 1: Secrets do Colab (recomendado)
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print('✅ API Key carregada dos Secrets do Colab!')
except:
    # Opção 2: Cole diretamente aqui
    GROQ_API_KEY = 'COLE_SUA_GROQ_API_KEY_AQUI'
    print('⚠️ Usando chave manual.')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY


## 3. Base de Conhecimento do Hotel (RAG)


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

BASE_CONHECIMENTO = [
    {"categoria": "geral", "pergunta": "Qual o horário de check-in e check-out?",
     "resposta": "O check-in é a partir das 14h e o check-out até as 12h. Early check-in e late check-out estão sujeitos a disponibilidade e podem ter custo adicional de R$80,00 por hora."},
    {"categoria": "geral", "pergunta": "Qual o endereço do hotel?",
     "resposta": "O Hotel LuxStay está localizado na Av. Beira Mar, 1500 - Centro, Florianópolis/SC. Próximo ao Shopping Beiramar e a 15 minutos do Aeroporto Hercílio Luz."},
    {"categoria": "geral", "pergunta": "Quais formas de pagamento são aceitas?",
     "resposta": "Aceitamos cartões de crédito (Visa, Mastercard, Elo, Amex), débito, PIX e dinheiro. Para reservas online, aceitamos cartão de crédito e PIX."},
    {"categoria": "geral", "pergunta": "O hotel aceita animais de estimação?",
     "resposta": "Sim! Somos pet-friendly. Aceitamos animais de até 15kg com taxa adicional de R$50,00 por noite. É necessário informar no momento da reserva."},
    {"categoria": "geral", "pergunta": "Qual a política de cancelamento?",
     "resposta": "Cancelamentos com mais de 48h de antecedência são gratuitos. Entre 24h e 48h, cobramos 50% da primeira diária. Com menos de 24h, cobramos a primeira diária integral."},
    {"categoria": "quartos", "pergunta": "Quais tipos de quarto estão disponíveis?",
     "resposta": "Oferecemos: Quarto Standard (R$280/noite) - cama casal, TV, ar-condicionado, Wi-Fi; Quarto Superior (R$380/noite) - cama king, varanda, frigobar, TV 50pol; Suíte Luxo (R$550/noite) - sala de estar, banheira, vista mar, minibar completo; Suíte Presidencial (R$950/noite) - 2 ambientes, jacuzzi, vista panorâmica, mordomo."},
    {"categoria": "quartos", "pergunta": "Os quartos possuem Wi-Fi?",
     "resposta": "Sim, todos os quartos e áreas comuns possuem Wi-Fi gratuito de alta velocidade (200 Mbps). A senha é fornecida no check-in."},
    {"categoria": "quartos", "pergunta": "Posso solicitar cama extra?",
     "resposta": "Sim, camas extras estão disponíveis por R$80,00 por noite, sujeito à disponibilidade e ao tipo de quarto. Berços para bebês são gratuitos."},
    {"categoria": "quartos", "pergunta": "O quarto tem cofre?",
     "resposta": "Sim, todos os quartos possuem cofre digital individual para guardar objetos de valor. O hotel não se responsabiliza por itens não guardados no cofre."},
    {"categoria": "servicos", "pergunta": "O hotel tem piscina?",
     "resposta": "Sim! Temos piscina ao ar livre com vista para o mar (aberta das 7h às 22h), piscina aquecida coberta (aberta das 6h às 23h) e piscina infantil."},
    {"categoria": "servicos", "pergunta": "Tem academia ou spa?",
     "resposta": "Sim! Nossa academia funciona 24h com equipamentos modernos. O Spa LuxStay oferece massagens, sauna, banho turco e tratamentos faciais (agendamento na recepção ou ramal 200)."},
    {"categoria": "servicos", "pergunta": "O café da manhã está incluso?",
     "resposta": "O café da manhã buffet está incluso em todas as tarifas. Servido das 6h30 às 10h30 (fins de semana até 11h). Temos opções veganas, sem glúten e sem lactose."},
    {"categoria": "servicos", "pergunta": "Tem estacionamento?",
     "resposta": "Sim, oferecemos estacionamento coberto com manobrista por R$40,00/dia para hóspedes. Vagas para veículos elétricos com carregador disponíveis."},
    {"categoria": "servicos", "pergunta": "Quais restaurantes o hotel possui?",
     "resposta": "Temos o Restaurante Mar Azul (almoço e jantar, cozinha contemporânea), o Lounge Bar (drinks e petiscos, 17h-01h) e o serviço de room service disponível 24h."},
    {"categoria": "servicos", "pergunta": "O hotel oferece transfer do aeroporto?",
     "resposta": "Sim! Oferecemos transfer aeroporto-hotel por R$90,00 (sedan) ou R$150,00 (van para grupos). Agende com 24h de antecedência pelo WhatsApp ou e-mail."},
    {"categoria": "servicos", "pergunta": "Tem lavanderia?",
     "resposta": "Sim, nosso serviço de lavanderia funciona de segunda a sábado. Roupas entregues até 9h ficam prontas no mesmo dia. Temos também lavanderia self-service no 2º andar."},
    {"categoria": "servicos", "pergunta": "O hotel tem sala de reuniões ou eventos?",
     "resposta": "Sim! Dispomos de 3 salas de reunião (10 a 50 pessoas) e 1 salão de eventos (até 200 pessoas). Equipamentos audiovisuais, coffee break e suporte técnico inclusos."},
    {"categoria": "turismo", "pergunta": "Quais praias ficam próximas ao hotel?",
     "resposta": "As praias mais próximas são: Praia Central (5 min a pé), Praia do Müller (10 min de carro), Praia de Jurerê (25 min de carro) e Praia da Joaquina (30 min de carro)."},
    {"categoria": "turismo", "pergunta": "O hotel oferece passeios turísticos?",
     "resposta": "Sim! Na recepção organizamos: city tour por Florianópolis, passeio de barco pela Baía Norte, trilha ecológica na Lagoa do Peri e visita ao centro histórico."},
    {"categoria": "turismo", "pergunta": "Onde posso alugar um carro?",
     "resposta": "Temos parceria com a locadora MoveCar, com balcão no lobby do hotel. Também há locadoras no aeroporto. A recepção pode ajudar a fazer a reserva."},
    {"categoria": "politicas", "pergunta": "Qual a idade mínima para hospedagem?",
     "resposta": "Menores de 18 anos devem estar acompanhados de um responsável legal. A partir de 16 anos, é necessário apresentar autorização registrada em cartório."},
    {"categoria": "politicas", "pergunta": "É permitido fumar no hotel?",
     "resposta": "O Hotel LuxStay é 100% livre de fumo em áreas internas. Temos área externa designada para fumantes no terraço do 1º andar."},
    {"categoria": "politicas", "pergunta": "Existe taxa para visitantes?",
     "resposta": "Visitantes podem acessar o lobby e restaurantes sem custo. Para usar piscina e academia, há taxa de day-use de R$120,00 por pessoa."},
    {"categoria": "contato", "pergunta": "Como entrar em contato com o hotel?",
     "resposta": "Telefone: (48) 3333-1500 | WhatsApp: (48) 99999-1500 | E-mail: reservas@luxstay.com.br | Instagram: @hotelluxstay. Recepção 24h para hóspedes."},
    {"categoria": "contato", "pergunta": "O hotel tem serviço de concierge?",
     "resposta": "Sim! Nosso concierge está disponível das 7h às 23h para reservas em restaurantes, ingressos para eventos, dicas de passeios e qualquer necessidade especial."},
]

print(f"✅ Base de conhecimento carregada: {len(BASE_CONHECIMENTO)} documentos")


## 4. Motor de Busca RAG (TF-IDF + Similaridade)


In [ ]:
# Preparar documentos para indexação
documentos = []
for item in BASE_CONHECIMENTO:
    texto = f"{item['pergunta']} {item['resposta']}"
    documentos.append({
        "texto_busca": texto,
        "resposta": item["resposta"],
        "categoria": item["categoria"],
        "pergunta": item["pergunta"],
    })

# Criar índice TF-IDF
textos = [doc["texto_busca"] for doc in documentos]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
tfidf_matrix = vectorizer.fit_transform(textos)

def buscar_contexto(pergunta, top_k=3):
    """Busca os documentos mais relevantes usando similaridade de cosseno."""
    emb_pergunta = vectorizer.transform([pergunta])
    similaridades = cosine_similarity(emb_pergunta, tfidf_matrix).flatten()
    indices = np.argsort(similaridades)[::-1][:top_k]

    contexto_partes = []
    for i in indices:
        doc = documentos[i]
        score = similaridades[i]
        if score > 0.05:
            contexto_partes.append(
                f"[{doc['categoria'].upper()}] P: {doc['pergunta']}\n"
                f"R: {doc['resposta']}"
            )

    if not contexto_partes:
        return "Nenhuma informação relevante encontrada."
    return "\n\n".join(contexto_partes)

# Teste do RAG
print("🔍 Teste do RAG:")
print("-" * 50)
resultado = buscar_contexto("Qual o horário da piscina?")
print(resultado)
print("\n✅ Motor RAG funcionando!")


## 5. Módulo de Tradução (8 idiomas)


In [ ]:
from deep_translator import GoogleTranslator

IDIOMAS = {
    "pt": "Português",
    "en": "English",
    "es": "Español",
    "fr": "Français",
    "de": "Deutsch",
    "it": "Italiano",
    "ja": "日本語",
    "zh-CN": "中文",
}

def traduzir_texto(texto, idioma_destino):
    """Traduz texto para o idioma selecionado."""
    if idioma_destino == "pt":
        return texto
    try:
        tradutor = GoogleTranslator(source="pt", target=idioma_destino)
        return tradutor.translate(texto)
    except Exception as e:
        print(f"⚠️ Erro na tradução: {e}")
        return texto

# Teste
print("🌐 Teste de tradução:")
print("-" * 50)
frase = "Bem-vindo ao Hotel LuxStay! O check-in é a partir das 14h."
print(f"PT: {frase}")
print(f"EN: {traduzir_texto(frase, 'en')}")
print(f"ES: {traduzir_texto(frase, 'es')}")
print("\n✅ Tradução funcionando!")


## 6. Módulo de Áudio (Texto → Voz)


In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
import tempfile

def gerar_audio(texto, idioma="pt"):
    """Gera áudio a partir de texto com gTTS."""
    with tempfile.NamedTemporaryFile(delete=False, suffix=".mp3") as tmp:
        tts = gTTS(text=texto, lang=idioma)
        tts.save(tmp.name)
        return tmp.name

# Teste
print("🔊 Gerando áudio de teste...")
audio_path = gerar_audio("Bem-vindo ao Hotel LuxStay! Como posso ajudá-lo?")
print("✅ Clique play abaixo para ouvir:")
display(Audio(audio_path, autoplay=False))


## 7. Módulo de Transcrição (Áudio → Texto com Whisper)

O Whisper transcreve áudio em qualquer idioma para texto.


In [ ]:
import whisper
from google.colab import files

print("⏳ Carregando modelo Whisper...")
modelo_whisper = whisper.load_model("base")
print("✅ Modelo Whisper pronto!")

def transcrever_audio(caminho_audio):
    """Transcreve áudio usando Whisper."""
    resultado = modelo_whisper.transcribe(caminho_audio)
    return resultado["text"]


## 8. Geração de Resposta com LLM (Groq + Llama 3.3)


In [ ]:
from groq import Groq

def gerar_resposta(pergunta, contexto):
    """Gera resposta usando Groq API com contexto do RAG."""
    prompt = f"""Você é o concierge virtual do Hotel LuxStay, um hotel de luxo em Florianópolis.
Seja educado, profissional e acolhedor. Use SOMENTE as informações abaixo para responder.
Se não tiver a informação, sugira que o hóspede entre em contato com a recepção pelo telefone (48) 3333-1500.

Contexto do Hotel:
{contexto}

Pergunta do hóspede:
{pergunta}

Responda de forma clara, objetiva e hospitaleira."""

    client = Groq(api_key=os.environ['GROQ_API_KEY'])
    chat = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile",
    )
    return chat.choices[0].message.content

print("✅ Módulo LLM configurado!")


---
## 9. 🤖 CHATBOT COMPLETO - Interaja aqui!

### 💬 Pergunta por Texto
Mude a variável `pergunta` e execute a célula.


In [ ]:
# ============================================
# ✏️ MUDE SUA PERGUNTA AQUI:
pergunta = "Qual o horário do café da manhã?"

# 🌐 MUDE O IDIOMA DA RESPOSTA AQUI (pt, en, es, fr, de, it, ja, zh-CN):
idioma_resposta = "pt"
# ============================================

print("🏨 Hotel LuxStay - Concierge Virtual")
print("=" * 55)
print(f"🧑 Hóspede: {pergunta}")
print(f"🌐 Idioma: {IDIOMAS[idioma_resposta]}")
print("-" * 55)

# 1. RAG - Buscar contexto
contexto = buscar_contexto(pergunta)
print("📚 Contexto RAG recuperado!")

# 2. LLM - Gerar resposta
resposta_pt = gerar_resposta(pergunta, contexto)

# 3. Tradução (se necessário)
if idioma_resposta != "pt":
    resposta_final = traduzir_texto(resposta_pt, idioma_resposta)
    print(f"🌐 Traduzido para {IDIOMAS[idioma_resposta]}")
else:
    resposta_final = resposta_pt

# 4. Exibir resposta
print("-" * 55)
print(f"🤖 Concierge: {resposta_final}")
print("-" * 55)

# 5. Gerar áudio
audio_path = gerar_audio(resposta_final, idioma_resposta)
print("🔊 Resposta em áudio:")
display(Audio(audio_path, autoplay=False))


### 🎙️ Pergunta por Áudio

Faça upload de um áudio e o sistema transcreve + responde.


In [ ]:
print("📁 Faça upload de um arquivo de áudio (mp3, wav, m4a, ogg):")
uploaded = files.upload()

if uploaded:
    nome_arquivo = list(uploaded.keys())[0]

    # 1. Transcrever
    print(f"\n🎙️ Transcrevendo: {nome_arquivo}...")
    transcricao = transcrever_audio(nome_arquivo)
    print(f"📝 Transcrição: {transcricao}")
    print("-" * 55)

    # 2. RAG
    contexto = buscar_contexto(transcricao)

    # 3. LLM
    resposta_pt = gerar_resposta(transcricao, contexto)

    # 4. Tradução
    if idioma_resposta != "pt":
        resposta_final = traduzir_texto(resposta_pt, idioma_resposta)
    else:
        resposta_final = resposta_pt

    # 5. Resposta
    print(f"🤖 Concierge: {resposta_final}")
    print("-" * 55)

    # 6. Áudio
    audio_path = gerar_audio(resposta_final, idioma_resposta)
    print("🔊 Resposta em áudio:")
    display(Audio(audio_path, autoplay=False))


### 🔄 Modo Conversa (várias perguntas seguidas)


In [ ]:
print("🏨 Hotel LuxStay - Modo Conversa")
print("Digite 'sair' para encerrar\n")

while True:
    pergunta = input("🧑 Você: ")

    if pergunta.lower() in ['sair', 'exit', 'quit']:
        print("\n👋 Obrigado! Tenha uma ótima estadia!")
        break

    if not pergunta.strip():
        continue

    contexto = buscar_contexto(pergunta)
    resposta_pt = gerar_resposta(pergunta, contexto)

    if idioma_resposta != "pt":
        resposta_final = traduzir_texto(resposta_pt, idioma_resposta)
    else:
        resposta_final = resposta_pt

    print(f"🤖 Concierge: {resposta_final}")
    audio_path = gerar_audio(resposta_final, idioma_resposta)
    display(Audio(audio_path, autoplay=False))
    print()


---
## 📋 Resumo Técnico

| Componente | Tecnologia | Função |
|------------|-----------|--------|
| Transcrição | OpenAI Whisper | Áudio → Texto |
| RAG | TF-IDF + Cosine Similarity | Busca na base de conhecimento |
| LLM | Groq (Llama 3.3 70B) | Geração de resposta natural |
| Tradução | Google Translator | Suporte a 8 idiomas |
| Áudio | gTTS | Texto → Voz |

### Fluxo:
```
Hóspede (texto/áudio) → [Whisper] → Texto → [RAG] → Contexto → [LLM] → Resposta → [Tradução] → [gTTS] → Áudio
```
